<a href="https://colab.research.google.com/github/oisiipasuta/signate_spectral_moisture/blob/main/%E8%BF%91%E8%B5%A4%E5%A4%96%E5%87%A6%E7%90%86%E5%AE%9F%E9%A8%93%E3%81%BE%E3%81%A8%E3%82%81.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
from google.colab import drive
drive.mount('/content/drive')
import pandas as pd
import numpy as np
import seaborn as sns
from matplotlib.pyplot import figure
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [15]:
df_train = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/SIGNATE/近赤外：木材の含水率推定/train (1).csv', encoding='cp932')
df_test = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/SIGNATE/近赤外：木材の含水率推定/test (1).csv', encoding='cp932')

In [16]:
df_train.head()

,sample number,species number,樹種,含水率,9993.76781,9989.9107,9986.05359,9982.19648,9978.33937,9974.48227,...,4034.53536,4030.67826,4026.82115,4022.96404,4019.10693,4015.24982,4011.39271,4007.5356,4003.6785,3999.82139
0,1,1,イチョウ,216.129032,0.41485,0.41465,0.41463,0.41476,0.41481,0.41470,...,1.25104,1.24925,1.24145,1.23620,1.23384,1.22981,1.22818,1.23087,1.23354,1.23219
1,2,1,イチョウ,210.752688,0.42049,0.42040,0.42049,0.42053,0.42038,0.42010,...,1.21929,1.21611,1.21565,1.21745,1.21680,1.21205,1.21074,1.21508,1.21901,1.21846
2,3,1,イチョウ,205.913979,0.41040,0.41045,0.41047,0.41028,0.41000,0.40989,...,1.17471,1.17147,1.16611,1.16633,1.16998,1.16955,1.16200,1.15341,1.15139,1.15403
3,4,1,イチョウ,201.075269,0.40080,0.40061,0.40031,0.40008,0.39993,0.39991,...,1.11271,1.11339,1.11115,1.10965,1.11017,1.11165,1.11426,1.11787,1.11849,1.11328
4,5,1,イチョウ,196.236559,0.38792,0.38814,0.38825,0.38817,0.38798,0.38778,...,1.05617,1.05596,1.05760,1.06014,1.06240,1.06566,1.06962,1.07056,1.06459,1.05612


In [112]:
df_train_metric = df_train.iloc[:, 3:]
df_test_metric = df_test.iloc[:, 3:]

X = df_train_metric.iloc[:, 1:]
y = df_train_metric.iloc[:, 0]
groups = df_train['species number'].values

display(X.head())
display(y.head())

,9993.76781,9989.9107,9986.05359,9982.19648,9978.33937,9974.48227,9970.62516,9966.76805,9962.91094,9959.05383,...,4034.53536,4030.67826,4026.82115,4022.96404,4019.10693,4015.24982,4011.39271,4007.5356,4003.6785,3999.82139
0,0.41485,0.41465,0.41463,0.41476,0.41481,0.41470,0.41452,0.41427,0.41392,0.41364,...,1.25104,1.24925,1.24145,1.23620,1.23384,1.22981,1.22818,1.23087,1.23354,1.23219
1,0.42049,0.42040,0.42049,0.42053,0.42038,0.42010,0.41988,0.41966,0.41942,0.41932,...,1.21929,1.21611,1.21565,1.21745,1.21680,1.21205,1.21074,1.21508,1.21901,1.21846
2,0.41040,0.41045,0.41047,0.41028,0.41000,0.40989,0.40992,0.40982,0.40951,0.40920,...,1.17471,1.17147,1.16611,1.16633,1.16998,1.16955,1.16200,1.15341,1.15139,1.15403
3,0.40080,0.40061,0.40031,0.40008,0.39993,0.39991,0.39998,0.39995,0.39967,0.39931,...,1.11271,1.11339,1.11115,1.10965,1.11017,1.11165,1.11426,1.11787,1.11849,1.11328
4,0.38792,0.38814,0.38825,0.38817,0.38798,0.38778,0.38761,0.38740,0.38720,0.38712,...,1.05617,1.05596,1.05760,1.06014,1.06240,1.06566,1.06962,1.07056,1.06459,1.05612


,含水率
0,216.129032
1,210.752688
2,205.913979
3,201.075269
4,196.236559


In [114]:
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.model_selection import GroupKFold, GridSearchCV, cross_val_score
from scipy.signal import savgol_filter
from sklearn.cross_decomposition import PLSRegression
from sklearn.linear_model import ElasticNet

In [31]:
# NO.2 Raw + StandardScaler + Ridge]
#CV_score
#fold RMSE: [46.60997214 30.77150571 12.21368229 12.63510936 11.33215589]
#mean RMSE: 22.712485078190433
#std RMSE: 13.980900940961272
model = Pipeline([
    ("scaler", StandardScaler()),
    ("model", Ridge(alpha=314))
])

# scikit-learn 1.6以上なら shuffle / random_state が使える
gkf = GroupKFold(n_splits=5, shuffle=True, random_state=42)

scores = cross_val_score(
    model,
    X,
    y,
    scoring="neg_root_mean_squared_error",
    cv=gkf,
    groups=groups,
    n_jobs=-1
)

rmse_scores = -scores
rmse_mean = rmse_scores.mean()
rmse_std = rmse_scores.std()

print("fold RMSE:", rmse_scores)
print("mean RMSE:", rmse_mean)
print("std RMSE:", rmse_std)

fold RMSE: [46.60997214 30.77150571 12.21368229 12.63510936 11.33215589]
mean RMSE: 22.712485078190433
std RMSE: 13.980900940961272


In [47]:
#NO.3 Raw + SNV + Ridge
#fold RMSE: [51.03274428 21.46247253 19.6002925  19.56788064  9.80690447]
#mean RMSE: 24.294058883578074
#std RMSE: 13.98013608472766

import numpy as np
from sklearn.base import BaseEstimator, TransformerMixin

class SNVTransformer(BaseEstimator, TransformerMixin):
    """
    Standard Normal Variate (SNV)

    各サンプル、つまり各スペクトルごとに
    平均0、標準偏差1に変換する前処理。

    X shape:
        (n_samples, n_wavelengths)
    """

    def fit(self, X, y=None):
        # SNVは各サンプル内で完結する処理なので、学習するパラメータはない
        return self

    def transform(self, X):
        X = np.asarray(X, dtype=float)

        mean = X.mean(axis=1, keepdims=True)
        std = X.std(axis=1, keepdims=True)

        # 標準偏差が0のサンプルがあった場合のゼロ除算対策
        std = np.where(std == 0, 1, std)

        return (X - mean) / std

model = Pipeline([
    ('SNV', SNVTransformer()),
    ("scaler", StandardScaler()),
    ("model", Ridge(alpha=10000))
])

gkf = GroupKFold(n_splits=5, shuffle=True, random_state=42)

scores = cross_val_score(
    model,
    X,
    y,
    scoring="neg_root_mean_squared_error",
    cv=gkf,
    groups=groups,
    n_jobs=-1
)

rmse_scores = -scores

print("fold RMSE:", rmse_scores)
print("mean RMSE:", rmse_scores.mean())
print("std RMSE:", rmse_scores.std())

fold RMSE: [51.03274428 21.46247253 19.6002925  19.56788064  9.80690447]
mean RMSE: 24.294058883578074
std RMSE: 13.98013608472766


In [71]:
# NO.4 Savitzky-Golay 1st derivative + StandardScaler + Ridge
#fold RMSE: [56.13456787 26.94373833 14.54905658 18.52527512 11.57787291]
#mean RMSE: 25.546102162221473
#std RMSE: 16.142775931489048

import numpy as np
from scipy.signal import savgol_filter

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.model_selection import GroupKFold, cross_val_score


class SavitzkyGolayTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, window_length=11, polyorder=2, deriv=1):
        self.window_length = window_length
        self.polyorder = polyorder
        self.deriv = deriv

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = np.asarray(X, dtype=float)

        return savgol_filter(
            X,
            window_length=self.window_length,
            polyorder=self.polyorder,
            deriv=self.deriv,
            axis=1
        )


model = Pipeline([
    ("sg", SavitzkyGolayTransformer(
        window_length=11,
        polyorder=3,
        deriv=1
    )),
    ("scaler", StandardScaler()),
    ("model", Ridge(alpha=100000))
])

gkf = GroupKFold(n_splits=5, shuffle=True, random_state=42)

scores = cross_val_score(
    model,
    X,
    y,
    scoring="neg_root_mean_squared_error",
    cv=gkf,
    groups=groups,
    n_jobs=-1
)

rmse_scores = -scores

print("fold RMSE:", rmse_scores)
print("mean RMSE:", rmse_scores.mean())
print("std RMSE:", rmse_scores.std())

fold RMSE: [56.13456787 26.94373833 14.54905658 18.52527512 11.57787291]
mean RMSE: 25.546102162221473
std RMSE: 16.142775931489048


In [91]:
# NO.5 SNV + Savitzky-Golay 1st derivative + StandardScaler + Ridge
#fold RMSE: [62.43372461 16.97709493 23.43247457 13.34727911 11.43811847]
#mean RMSE: 25.525738338303988
#std RMSE: 18.90204438902727

model = Pipeline([
    ("snv", SNVTransformer()),
    ("sg", SavitzkyGolayTransformer(
        window_length=11,
        polyorder=3,
        deriv=1
    )),
    ("scaler", StandardScaler()),
    ("model", Ridge(alpha=650))
])


gkf = GroupKFold(n_splits=5, shuffle=True, random_state=42)

scores = cross_val_score(
    model,
    X,
    y,
    scoring="neg_root_mean_squared_error",
    cv=gkf,
    groups=groups,
    n_jobs=-1
)

rmse_scores = -scores

print("fold RMSE:", rmse_scores)
print("mean RMSE:", rmse_scores.mean())
print("std RMSE:", rmse_scores.std())

fold RMSE: [62.43372461 16.97709493 23.43247457 13.34727911 11.43811847]
mean RMSE: 25.525738338303988
std RMSE: 18.90204438902727


In [111]:
# NO.6 SNV + Savitzky-Golay 1st derivative + StandardScaler + PLSR
#fold RMSE: [81.8936581  19.09949303 26.56615215 13.97620122 12.79966777]
#mean RMSE: 30.867034456766554
#std RMSE: 25.970825657201182
model = Pipeline([
    ("snv", SNVTransformer()),
    ("sg", SavitzkyGolayTransformer(
        window_length=11,
        polyorder=2,
        deriv=1
    )),
    ("scaler", StandardScaler()),
    ("model", PLSRegression(
        n_components=10,
        scale=False
    ))
])

gkf = GroupKFold(n_splits=5, shuffle=True, random_state=42)

scores = cross_val_score(
    model,
    X,
    y,
    scoring="neg_root_mean_squared_error",
    cv=gkf,
    groups=groups,
    n_jobs=-1
)

rmse_scores = -scores

print("fold RMSE:", rmse_scores)
print("mean RMSE:", rmse_scores.mean())
print("std RMSE:", rmse_scores.std())

fold RMSE: [81.8936581  19.09949303 26.56615215 13.97620122 12.79966777]
mean RMSE: 30.867034456766554
std RMSE: 25.970825657201182


In [ ]:
# Raw → StandardScaler → ElasticNet
from sklearn.metrics import mean_squared_error
pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", ElasticNet(
        max_iter=100000,
        random_state=42
    ))
])

param_grid = {
    "model__alpha": [0.001, 0.003, 0.01, 0.03, 0.1, 0.3, 1.0, 3.0, 10.0],
    "model__l1_ratio": [0.05, 0.1, 0.2, 0.5, 0.8, 0.95]
}

cv = GroupKFold(n_splits=5)

grid = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    scoring="neg_root_mean_squared_error",
    cv=cv,
    n_jobs=-1,
    verbose=1
)

grid.fit(X, y, groups=groups)

print("best params:", grid.best_params_)
print("best CV RMSE:", -grid.best_score_)

best_model = grid.best_estimator_

fold_rmse = []

for fold, (train_idx, valid_idx) in enumerate(cv.split(X, y, groups), start=1):
    X_train, X_valid = X.iloc[train_idx], X.iloc[valid_idx]
    y_train, y_valid = y.iloc[train_idx], y.iloc[valid_idx]

    best_model.fit(X_train, y_train)
    pred = best_model.predict(X_valid)

    rmse = mean_squared_error(y_valid, pred, squared=False)
    fold_rmse.append(rmse)

print("fold RMSE:", np.array(fold_rmse))
print("mean RMSE:", np.mean(fold_rmse))
print("std RMSE:", np.std(fold_rmse))
print("worst fold RMSE:", np.max(fold_rmse))

Fitting 5 folds for each of 54 candidates, totalling 270 fits


In [ ]:
# SNV → StandardScaler → ElasticNet
pipeline = Pipeline([
    ("snv", SNVTransformer()),
    ("scaler", StandardScaler()),
    ("model", ElasticNet(
        max_iter=100000,
        random_state=42
    ))
])

param_grid = {
    "model__alpha": [
        0.001, 0.003, 0.01, 0.03,
        0.1, 0.3, 1.0, 3.0, 10.0
    ],
    "model__l1_ratio": [
        0.05, 0.1, 0.2, 0.5, 0.8, 0.95
    ]
}

cv = GroupKFold(n_splits=5)

grid = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    scoring="neg_root_mean_squared_error",
    cv=cv,
    n_jobs=-1,
    verbose=1
)

grid.fit(X, y, groups=groups)

print("best params:", grid.best_params_)
print("best CV RMSE:", -grid.best_score_)

best_params = grid.best_params_

fold_rmse = []

for fold, (train_idx, valid_idx) in enumerate(cv.split(X, y, groups), start=1):
    X_train = X.iloc[train_idx] if hasattr(X, "iloc") else X[train_idx]
    X_valid = X.iloc[valid_idx] if hasattr(X, "iloc") else X[valid_idx]
    y_train = y.iloc[train_idx] if hasattr(y, "iloc") else y[train_idx]
    y_valid = y.iloc[valid_idx] if hasattr(y, "iloc") else y[valid_idx]

    model = Pipeline([
        ("snv", SNVTransformer()),
        ("scaler", StandardScaler()),
        ("model", ElasticNet(
            alpha=best_params["model__alpha"],
            l1_ratio=best_params["model__l1_ratio"],
            max_iter=100000,
            random_state=42
        ))
    ])

    model.fit(X_train, y_train)
    pred = model.predict(X_valid)

    rmse = mean_squared_error(y_valid, pred, squared=False)
    fold_rmse.append(rmse)

print("fold RMSE:", np.array(fold_rmse))
print("mean RMSE:", np.mean(fold_rmse))
print("std RMSE:", np.std(fold_rmse))
print("worst fold RMSE:", np.max(fold_rmse))

In [ ]:
# Raw → StandardScaler → PLSR
pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", PLSRegression(
        scale=False  # すでにStandardScalerを使うのでFalseにする
    ))
])

param_grid = {
    "model__n_components": [1, 2, 3, 5, 8, 10]
}

cv = GroupKFold(n_splits=5)

grid = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    scoring="neg_root_mean_squared_error",
    cv=cv,
    n_jobs=-1,
    verbose=1
)

grid.fit(X, y, groups=groups)

print("best params:", grid.best_params_)
print("best CV RMSE:", -grid.best_score_)

best_params = grid.best_params_

fold_rmse = []

for fold, (train_idx, valid_idx) in enumerate(cv.split(X, y, groups), start=1):
    X_train = X.iloc[train_idx] if hasattr(X, "iloc") else X[train_idx]
    X_valid = X.iloc[valid_idx] if hasattr(X, "iloc") else X[valid_idx]
    y_train = y.iloc[train_idx] if hasattr(y, "iloc") else y[train_idx]
    y_valid = y.iloc[valid_idx] if hasattr(y, "iloc") else y[valid_idx]

    model = Pipeline([
        ("scaler", StandardScaler()),
        ("model", PLSRegression(
            n_components=best_params["model__n_components"],
            scale=False
        ))
    ])

    model.fit(X_train, y_train)
    pred = model.predict(X_valid).ravel()

    rmse = np.sqrt(mean_squared_error(y_valid, pred))
    fold_rmse.append(rmse)

print("fold RMSE:", np.array(fold_rmse))
print("mean RMSE:", np.mean(fold_rmse))
print("std RMSE:", np.std(fold_rmse))
print("worst fold RMSE:", np.max(fold_rmse))

In [ ]:
# =========================
# SNV → StandardScaler → PLSR
# =========================

pipeline = Pipeline([
    ("snv", SNVTransformer()),
    ("scaler", StandardScaler()),
    ("model", PLSRegression(
        scale=False  # StandardScalerを使うのでFalse
    ))
])

param_grid = {
    "model__n_components": [1, 2, 3, 5, 8, 10]
}

cv = GroupKFold(n_splits=5)

grid = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    scoring="neg_root_mean_squared_error",
    cv=cv,
    n_jobs=-1,
    verbose=1
)

grid.fit(X, y, groups=groups)

print("best params:", grid.best_params_)
print("best CV RMSE:", -grid.best_score_)

best_params = grid.best_params_

fold_rmse = []

for fold, (train_idx, valid_idx) in enumerate(cv.split(X, y, groups), start=1):
    X_train = X.iloc[train_idx] if hasattr(X, "iloc") else X[train_idx]
    X_valid = X.iloc[valid_idx] if hasattr(X, "iloc") else X[valid_idx]
    y_train = y.iloc[train_idx] if hasattr(y, "iloc") else y[train_idx]
    y_valid = y.iloc[valid_idx] if hasattr(y, "iloc") else y[valid_idx]

    model = Pipeline([
        ("snv", SNVTransformer()),
        ("scaler", StandardScaler()),
        ("model", PLSRegression(
            n_components=best_params["model__n_components"],
            scale=False
        ))
    ])

    model.fit(X_train, y_train)
    pred = model.predict(X_valid).ravel()

    rmse = np.sqrt(mean_squared_error(y_valid, pred))
    fold_rmse.append(rmse)

print("fold RMSE:", np.array(fold_rmse))
print("mean RMSE:", np.mean(fold_rmse))
print("std RMSE:", np.std(fold_rmse))
print("worst fold RMSE:", np.max(fold_rmse))

In [ ]:
from sklearn.svm import SVR
# =========================
# Raw → StandardScaler → SVR
# =========================

pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", SVR(kernel="rbf"))
])

param_grid = {
    "model__C": [0.1, 0.3, 1.0, 3.0, 10.0, 30.0],
    "model__epsilon": [0.01, 0.03, 0.1, 0.3, 1.0],
    "model__gamma": ["scale", 0.001, 0.003, 0.01, 0.03]
}

cv = GroupKFold(n_splits=5)

grid = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    scoring="neg_root_mean_squared_error",
    cv=cv,
    n_jobs=-1,
    verbose=1
)

grid.fit(X_raw, y, groups=groups)

print("best params:", grid.best_params_)
print("best CV RMSE:", -grid.best_score_)

best_params = grid.best_params_

fold_rmse = []

for fold, (train_idx, valid_idx) in enumerate(cv.split(X, y, groups), start=1):
    X_train = X.iloc[train_idx] if hasattr(X, "iloc") else X[train_idx]
    X_valid = X.iloc[valid_idx] if hasattr(X, "iloc") else X[valid_idx]
    y_train = y.iloc[train_idx] if hasattr(y, "iloc") else y[train_idx]
    y_valid = y.iloc[valid_idx] if hasattr(y, "iloc") else y[valid_idx]

    model = Pipeline([
        ("scaler", StandardScaler()),
        ("model", SVR(
            kernel="rbf",
            C=best_params["model__C"],
            epsilon=best_params["model__epsilon"],
            gamma=best_params["model__gamma"]
        ))
    ])

    model.fit(X_train, y_train)
    pred = model.predict(X_valid)

    rmse = np.sqrt(mean_squared_error(y_valid, pred))
    fold_rmse.append(rmse)

print("fold RMSE:", np.array(fold_rmse))
print("mean RMSE:", np.mean(fold_rmse))
print("std RMSE:", np.std(fold_rmse))
print("worst fold RMSE:", np.max(fold_rmse))